# fMRI オープンデータ解析講習：安静時の脳のつながりから、ASD の人と定型の人を見分ける

01 では、課題をしているときに脳の **どこ** が反応したかを GLM で調べました。02 と 03 では、脳の活動のパターンから、
どちらの条件の課題を行っていたかを当てました（デコーディング）。ここでは、3 つのことが変わります。

- **課題がありません**。MRI の中で数分間、何もせずに過ごしているときの脳の活動（**安静時 fMRI**）を使います。
- 特徴は、脳の領域どうしの信号がどのくらい一緒に動くか（**機能的結合**）です。
- 当てるのは課題ではなく、**その人が自閉スペクトラム症（ASD）か、定型発達（以下、定型）か** です。

問いは、**脳のつながりから、ASD の人と定型の人を見分けられるか** です。世界の 17 の施設が持ち寄ったデータ **ABIDE I**
（871 人）で分類器を作り、学習に使っていない人を当てられるかを確かめます（5 章）。次に、手順を誤ると、当たっていないのに
当たったように見えてしまう例を見ます（6 章）。最後に、人数を集めることの意味を見ます（7 章）。

- **ゴール 5**：脳のつながりから ASD の人と定型の人を当て、それが偶然では説明しにくいことを確かめる（5・6 章）

**始める前に**：01〜03 のタブで、メニューの「ランタイム」→「ランタイムを接続解除して削除」を選んでから、このノートブックを
実行してください。Colab の無料枠では、同時に使えるメモリなどに限りがあるためです。

### このノートブックの読み方

左に縦線の付いていない文がメインの文章です。左に縦線が付いた「**くわしく**」は、時間があるときや、
あとで読むための部分で、講習の間は読み飛ばしてかまいません。コードは、1 行ずつ読めなくてかまいません。

> **くわしく**（時間があれば）
>
> - 02 を済ませていなくても、このノートブックだけで動きます。分類器や交差検証の説明は 02 より短くしてあります。
> - メインの文章だけを読めば、流れが分かるように書いてあります。
> - Colab での使い方は 01〜03 と同じです。セルを選んで Shift + Enter で、上から順に実行します。赤い文字（エラー）が出たら、
>   上のセルを飛ばしていないかを確かめ、メニューの「ランタイム」→「すべてのセルを実行」を試してください。
> - 接続が切れると、取ってきたデータも消えます。そのときも「ランタイム」→「すべてのセルを実行」で、最初からやり直せます。
> - エラーの `NameError` は、たいてい上のセルを実行していないときに出ます。
> - GPU は使いません。
> - なぜ当てたいのか：1 つは、脳の計測が診断の手がかり（**バイオマーカー**）になるかを知るためです。もう 1 つは、当てるのに
>   使われた結合から、2 つの群で脳のどこが違うのかを探るためです。この 2 つは別の問いで、当たったからといって、どちらの
>   答えにもそのままなるわけではありません（8 章）。

## 0. 準備

脳画像の表示のための Python のライブラリ **Nilearn** を入れます。01〜03 と同じバージョンです。
「restart the kernel」という表示が出ても、そのまま次へ進んでかまいません。

0 章の 2 つのセルは、続けて実行するだけでかまいません。

In [ ]:
%pip install -q nilearn==0.13.1

使う道具を読み込みます。

In [ ]:
import socket
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nilearn import plotting
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC

rng = np.random.default_rng(0)  # 乱数を使うところで、毎回同じ結果になるように
socket.setdefaulttimeout(60)  # 取得の途中で通信が 60 秒止まったら、やり直す

> **くわしく**（時間があれば）
>
> 分類器には、Colab に最初から入っている機械学習のライブラリ **scikit-learn** を使います。

## 1. データ：ABIDE I

**ABIDE I** は、世界の 17 の施設が、それまでに撮っていた ASD の人と定型の人の脳画像を持ち寄って公開したデータです
（Di Martino et al., 2014）。ここでは、それを **Preprocessed Connectomes Project（PCP）** が全員同じ手順で前処理した結果を
使います。利用は、非営利の目的に限られます。

まず、ファイルを取ってくる関数を作ります（実行するだけでかまいません）。

In [ ]:
base = "https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative"
data_dir = Path("data")


def download(url, path):
    """1 つのファイルを取ってくる。すでに取ってあれば取り直さない。通信が切れたら 3 回まで試す"""
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        part = path.with_name(path.name + ".part")  # 途中で切れても壊れたファイルが残らないように
        for attempt in range(3):
            try:
                urllib.request.urlretrieve(url, part)
                break
            except OSError:
                if attempt == 2:
                    raise
                time.sleep(2)
        part.rename(path)
    return path

参加した人たちの表（診断、年齢、性、施設、頭の動きなど）を取ってきます。1 行が 1 人です。
`DX_GROUP` が診断（1 が ASD、2 が定型）、`SITE_ID` が施設です。

In [ ]:
phenotypes = pd.read_csv(download(f"{base}/Phenotypic_V1_0b_preprocessed1.csv", data_dir / "phenotypes.csv"))
print(len(phenotypes), "people")
phenotypes[["SUB_ID", "SITE_ID", "DX_GROUP", "AGE_AT_SCAN", "SEX", "func_mean_fd"]].head()

そのうち、前処理済みのデータがあり、品質の確認を通った人だけを使います。

In [ ]:
passed = (
    (phenotypes["FILE_ID"] != "no_filename")  # 前処理済みのデータがある
    & (phenotypes["qc_rater_1"] == "OK")
    & phenotypes["qc_anat_rater_2"].isin(["OK", "maybe"])
    & phenotypes["qc_func_rater_2"].isin(["OK", "maybe"])
    & (phenotypes["qc_anat_rater_3"] == "OK")
    & (phenotypes["qc_func_rater_3"] == "OK")
)
people = phenotypes[passed].reset_index(drop=True)
print(len(people), "people passed the quality check")

> **くわしく**（時間があれば）
>
> - **ABIDE**（Autism Brain Imaging Data Exchange）の最初の ABIDE I には、ASD の人 539 人と定型の人 573 人（7〜64 歳）の、
>   安静時 fMRI と脳の形の画像が入っています。
> - PCP（Craddock et al., 2013）は、同じデータを 4 つの前処理の道具（CCS、C-PAC、DPARSF、NIAK）でそれぞれ処理した結果を
>   公開していて、ここでは C-PAC の結果を使います。01 の 1 章の図でいえば、前処理の結果を取ってきてそのまま解析する下の段と
>   同じ形ですが、置き場所は OpenNeuro ではなく PCP で、前処理の道具も fMRIPrep ではありません。
> - **利用の条件**：非営利の目的に限ること、出典を示すこと、手を加えたものを配るときは同じ条件にすること、です
>   （[Creative Commons 表示-非営利-継承 3.0](https://creativecommons.org/licenses/by-nc-sa/3.0/)。ABIDE のページの
>   [Usage Agreement](http://fcon_1000.projects.nitrc.org/indi/abide/abide_I.html)）。研究で使うときは、使った施設のデータと、
>   その研究費の出どころも記すよう求められています。ABIDE の本家（NITRC）から取るには登録が要りますが、ここで使う
>   前処理済みのデータは、公開の置き場（Amazon S3）から登録なしで取れます。
> - 品質の確認：PCP では、3 人の評価者が、元の画像（脳の形の画像と、機能画像）と前処理の結果を目で確かめています。
>   ここで使う人の選び方は、Nilearn の `fetch_abide_pcp` で `quality_checked=True` にしたときと同じ条件です。

## 2. データを取ってくる

1 人 1 つのファイル（約 0.4 MB）を、871 人分取ってきます。合わせて約 340 MB です。8 つずつ並行して取ってくるので、
数分で終わります。**待っている間に、3 章を読み進めてください。**

In [ ]:
strategy = "filt_noglobal"  # 前処理のバージョン。9 章で "filt_global" に変えて試せます


def roi_file(file_id):
    """1 人分の、領域ごとの信号のファイルについて、取ってくる元（URL）と、置き場所"""
    url = f"{base}/Outputs/cpac/{strategy}/rois_cc200/{file_id}_rois_cc200.1D"
    return url, data_dir / strategy / url.rsplit("/", 1)[1]


jobs = [roi_file(file_id) for file_id in people["FILE_ID"]]
with ThreadPoolExecutor(max_workers=8) as pool:  # 8 つずつ並行して取ってくる
    paths = list(pool.map(lambda job: download(*job), jobs))
print(len(paths), "files,", round(sum(p.stat().st_size for p in paths) / 1e6), "MB")

> **くわしく**（時間があれば）
>
> 前処理のバージョンは `filt_noglobal` を使います。PCP の前処理では、01 の 6 章で設計行列に頭の動きの列を入れたのと同じ考え方で、
> 脳の活動と関係のない変化を、信号から取り除いてあります。取り除いたのは、頭の動き（01 の 6 つをもとにした 24 個の値）、
> 神経の活動をほとんど映さない白質と脳脊髄液の信号から取り出した成分（5 個）、ゆっくりした変化（1 次と 2 次の傾き）で
> 説明できる変化で、どのバージョンでも同じです。`filt_noglobal` はそれに加えて、0.01〜0.1 Hz のゆっくりした揺らぎだけを残し
> （帯域フィルタ）、脳全体の平均の信号は取り除かないバージョンです。

## 3. 参加した人たち

`group` という列を作り、ASD の人を `ASD`、定型の人を `TD`（typically developing の略。03 の `NT` と同じく、定型の人）とします。群ごとの人数、年齢、男性の割合、知能指数（全検査 IQ）です。

In [ ]:
people["group"] = np.where(people["DX_GROUP"] == 1, "ASD", "TD")
people.groupby("group").agg(
    people=("SUB_ID", "size"),
    mean_age=("AGE_AT_SCAN", "mean"),
    min_age=("AGE_AT_SCAN", "min"),
    max_age=("AGE_AT_SCAN", "max"),
    male=("SEX", lambda sex: (sex == 1).mean()),  # 1 が男性
    mean_IQ=("FIQ", lambda iq: iq[iq > 0].mean()),  # 記録が無い人（-9999）を除く
).round(2)

ASD の人が 403 人、定型の人が 468 人です。6〜58 歳と幅が広く、8 割以上が男性です。結果は、こうした人たちについてのものです。

> **くわしく**（時間があれば）
>
> - ABIDE のページには 7〜64 歳とありますが、表には 6.47 歳の人もいます。公開データでは、こうした小さな食い違いがよくあります。
> - ABIDE I が集められた頃の診断の基準（DSM-IV-TR）では、今の ASD は、自閉性障害、アスペルガー障害、特定不能の広汎性発達障害
>   に分かれていて、ASD の群にはそのどれもが入っています。細かい診断の列（`DSM_IV_TR`）は、OLIN という施設では群に関係なく
>   全員が 0（control）と記録されています。施設によって記録の付け方が違うことがあるので、診断には `DX_GROUP` を使います。

## 4. 1 人分の特徴：脳のつながり

PCP では、脳を **200 の領域** に分けて（**CC200** という地図）、領域ごとに信号を平均しています。例として 1 人分を読み込みます。
1 行が 1 枚の画像（時刻）、1 列が 1 つの領域です。

In [ ]:
fd = people["func_mean_fd"].to_numpy()  # 撮像中の頭の動きの大きさ（平均 FD）
example = int(np.argmin(np.abs(fd - np.median(fd))))  # 頭の動きが全員の中央値にいちばん近い人
time_series = np.loadtxt(paths[example])
print(people.loc[example, ["SITE_ID", "group", "AGE_AT_SCAN", "func_mean_fd"]].to_dict())
print("table (volumes, regions):", time_series.shape)

**機能的結合** は、2 つの領域の信号がどのくらい一緒に上がり下がりするかを表す **相関係数** です（1 に近いほど一緒に動き、
0 なら関係がありません）。200 の領域のすべての組で計算すると、200 × 200 の表（**結合行列**）になります。

In [ ]:
matrix = np.corrcoef(time_series.T)
plotting.plot_matrix(matrix, vmin=-1, vmax=1, colorbar=True, reorder=True,
                     labels=[" "] * 200,  # 並べ替えには名前が要るが、200 個は読めないので空白にする
                     title="functional connectivity of one person (reordered)")
plotting.show()

赤い四角の塊は、一緒に動く領域のまとまりです。ただし、相関が高くても、直接つながっているとは限りません（頭の動きのような
雑音が両方に入っていても、相関は高くなります）。

表は対角線をはさんで対称なので、右上の三角（200 × 199 ÷ 2 ＝ **19,900 個**）を、その人の特徴にします。ここからは、
02 のボクセルの値と同じように **1 人 ＝ 19,900 個の数の並び** として扱い、どの結合が何を表すかには立ち入りません。
次のセルは、実行するだけでかまいません。

In [ ]:
upper = np.triu_indices(200, k=1)  # 行列の右上の三角（対角線より上）の位置


def connectivity_features(time_series):
    """1 人分の信号から、19,900 個の結合の強さ（Fisher の z）を返す"""
    with np.errstate(invalid="ignore", divide="ignore"):  # 信号がずっと 0 の領域があっても止めない
        matrix = np.corrcoef(time_series.T)
    return np.arctanh(np.nan_to_num(matrix[upper]))  # 計算できない結合（NaN）は 0 にする

全員分の特徴を作り、1 行が 1 人、1 列が 1 つの結合の表 `X` にまとめます。当てる答え（ASD か定型か）は `y` にまとめます。10 秒ほどかかります。

In [ ]:
features, n_flat = [], []
for path in paths:
    time_series = np.loadtxt(path)
    n_flat.append(int((time_series.std(axis=0) == 0).sum()))
    features.append(connectivity_features(time_series))

X = np.vstack(features)
y = people["group"].to_numpy()  # 当てる答え（ASD か TD か）
print("X (people, connections):", X.shape)
print("people with flat regions:", sum(n > 0 for n in n_flat), "/ at most", max(n_flat), "regions")

> **くわしく**（時間があれば）
>
> - CC200（Craddock et al., 2012）は、安静時の信号がよく似た近くのボクセルどうしをまとめて作った地図です。
> - 結合行列の図は、似た動きをする領域どうしが近くに来るように、行と列を並べ替えています（`reorder=True`）。赤い塊は、
>   安静時に一緒に動く領域のまとまりで、**ネットワーク** と呼ばれます（視覚、運動、デフォルトモードなど）。
> - 相関は、直接の配線や、情報が流れる向きを表すものではありません。共通の別の領域から信号を受けていても高くなります。
> - 相関係数は −1〜1 に収まり、±1 の近くで値が詰まるので、**Fisher の z 変換**（arctanh）で広げてから使っています。
> - あわせて、信号がずっと 0 の領域の数も数えています。15 人では、信号がずっと 0 の領域が 1〜12 個あります。
>   この教材を作るときに調べたところ、側頭葉の下の部分、眼窩前頭皮質、小脳など、撮像の範囲から外れやすい場所や、信号が
>   落ちやすい場所でした。そうした領域との結合は計算できないので、0（相関なし）として扱います。

## 5. ASD か定型かを当てる

02 と同じ **線形 SVM** で、その人が ASD か定型かを当てます。設定の 1 つ（**C**）は、この教材を作るときに試して選んだ値
（0.001）です（6 章の最後でふれます）。

**考えてみましょう**：脳のつながりから、ASD の人か定型の人かを、どのくらい当てられると思いますか。

次のセルは、分類器の用意です（実行するだけでかまいません）。

In [ ]:
K = X @ X.T  # 871 × 871 の内積の表
svm = SVC(kernel="precomputed", C=0.001)

> **くわしく**（時間があれば）
>
> - 02 と同じく、先に人どうしの内積の表（871 × 871）を作っておくと、速く計算できます。
> - **C** は、学習のときに、学習に使った例を間違えることをどれだけ嫌うかの設定です。大きいほど、学習の例をなるべく間違えない
>   ように重みを決め、小さいほど、少しの間違いは許す代わりに、重みを小さく保ちます。02 では 1.0 でした。ここでは、特徴が
>   19,900 個と多いので、ずっと小さい 0.001 にして、1 つ 1 つの結合に強く頼りすぎないようにします。

871 人を 10 組に分け、1 組をテスト用に残して残りの 9 組で学習し、残した組の人を当てます。これを 10 組すべてについて
繰り返します（**10 分割の交差検証**）。どの人も、その人を学習に使っていない分類器で当てることになります。

2 つの群の人数が違うので、多いほう（定型）を答え続けるだけで 0.54 当たります。これが比べる相手です。

In [ ]:
ten_fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)  # 2 つの群の割合をそろえて、10 組に分ける
predicted = cross_val_predict(svm, K, y, cv=ten_fold)
accuracy = (predicted == y).mean()
print("accuracy (10-fold cross-validation):", accuracy.round(3))
print("always answering the larger group (TD):", (y == "TD").mean().round(3))
pd.crosstab(pd.Series(y, name="truth"), pd.Series(predicted, name="predicted"))

10 分割の交差検証で 0.692 当たり、多いほうを答え続ける 0.54 を上回りました。表は、行が本当の群、列が分類器の答えです。
ASD の 403 人のうち 251 人（0.62）を ASD と当て、定型の 468 人のうち 352 人（0.75）を定型と当てました。ASD の人のほうが、
少し見逃されやすくなっています。

偶然でもこのくらい当たるかを、02・03 と同じく **並べ替え検定** で確かめます。診断のラベルを混ぜて同じことを 200 回繰り返し、
混ぜたラベルでどのくらい当たるかと比べます。1 分ほどかかります。

In [ ]:
site = people["SITE_ID"].str.replace(r"_[12]$", "", regex=True).to_numpy()  # 撮った施設（UM_1 と UM_2 を UM に、などとまとめる）


def shuffle_within_site(labels):
    """同じ施設の人の中だけで、ラベルを混ぜる"""
    shuffled = labels.copy()
    for s in np.unique(site):
        rows = np.where(site == s)[0]
        shuffled[rows] = labels[rng.permutation(rows)]
    return shuffled


def ten_fold_accuracy(labels):
    """10 分割の交差検証で当て、正解率を返す"""
    return (cross_val_predict(svm, K, labels, cv=ten_fold) == labels).mean()


n_permutations = 200  # 混ぜる回数。時間が足りないときは減らせます
null_accuracy = np.array([ten_fold_accuracy(shuffle_within_site(y)) for _ in range(n_permutations)])
p_value = (np.sum(null_accuracy >= accuracy) + 1) / (len(null_accuracy) + 1)
print("shuffled labels: mean", null_accuracy.mean().round(3), "/ max", null_accuracy.max().round(3))
print("p =", round(p_value, 4))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(null_accuracy, bins=20, color="gray", label=f"shuffled labels ({n_permutations} times)")
ax.axvline(accuracy, color="red", label=f"true labels ({accuracy:.2f})")
ax.set_xlabel("accuracy")
ax.set_xlim(0.3, 1)
ax.legend(loc="upper right")
plt.show()

混ぜたラベルの正解率は 0.5 のまわりに集まり、本物はどれよりも上です（p は約 0.005。200 回で出せるいちばん小さい値）。

**02 と比べると**：02 の 0.96 は、同じ人の中の、はっきり違う 2 つの状態の比べでした。ここでは人と人を比べます。人と人は、
ASD かどうかのほかにも、年齢や性などで大きく違います。それでも、学習に使っていない人に、偶然よりはっきり高く
当たりました。脳のつながりに、ASD の人と定型の人で、**小さいけれども偶然ではない違い** があると考えられます。ただし、
この成績だけでは、診断に役立つかは判断できません。

> **くわしく**（時間があれば）
>
> - p 値は、「混ぜたのに本物と同じかそれ以上当たった回数 ＋ 1」を「混ぜた回数 ＋ 1」で割ったものです（本物の並べ方も、
>   偶然ありえた並べ方の 1 つとして数えるため、＋ 1 します。Ojala & Garriga, 2010）。200 回なら、いちばん小さい p 値は
>   1 ÷ 201 で約 0.005 です。混ぜる回数を増やせば、もっと小さい p 値を示せるかもしれません。
> - ラベルは、同じ施設の人の中だけで混ぜています。施設によって ASD の人の割合が違う（0.30〜0.64）ので、全体で混ぜると、
>   施設の違いを手がかりに当たる分まで「偶然」の側から消えてしまいます。施設の中で混ぜれば、その分は混ぜたあとにも残るので、
>   本物がそれを上回れば、施設の違いだけでは説明しにくいと言えます。
>   施設は、20 通りの `SITE_ID` のうち、データが 2 組に分かれている 3 つ（`UM_1` と `UM_2` など）をまとめて、17 としています。
> - **施設や頭の動きで当たっていないか**：この教材を作るときに確かめました。施設ごとに装置や撮る長さなどが違うので、同じ結合から
>   撮った施設を当てると、17 択で約 0.75 当たります。そこで、1 つの施設を丸ごと除いて学習し、除いた施設の人を当てる交差検証も
>   しましたが、正解率は 0.689 で、ほとんど変わりませんでした。また、ASD の人のほうが、撮像中の頭の動きが少し大きく（1 枚撮るごとの
>   動きの平均で 0.126 mm と 0.094 mm）、頭の動きは、前処理のあとも結合の強さを系統的に変えることが知られています
>   （Power et al., 2012）。ただし、脳を使わずに、頭の動き・年齢・性だけで当てると 0.565 で、脳のつながりにはだいぶ届きませんでした。
> - 結合を 1 本ずつ、2 つの群の平均で比べると、差はとても小さく（効果量 d は最大 0.45 くらい。この教材を作るときに調べた値）、
>   ASD の人のほうが弱い結合が多いという報告（Di Martino et al., 2014）と合う向きでした。分類器は、19,900 本の小さな差を
>   少しずつ足し合わせて判定しています。どの結合が違うかの解釈は、安静時の結合を扱う論文に譲ります。
> - 診断に役立つかは、どんな人に、どんな場面で使い、どの間違いがどれだけ困るかによります。正解率だけでは決まりません。
> - これまでの研究でも、ABIDE I の多施設のデータでは 0.6〜0.7 くらいです（Nielsen et al., 2013 で約 0.60、同じ 871 人を
>   使った Abraham et al., 2017 で約 0.67、深層学習を使った Heinsfeld et al., 2018 で約 0.70）。

## 6. 学習の外で答えを見てしまう（リーク）

5 章では、テストする人の答えを学習に使わないように、交差検証で当てました。この手順を少し誤ると、何が起こるかを見ます。

19,900 個は多いので、「ASD と定型で差の大きい結合を 500 個選んでから学習しよう」と考えたとします。このとき、
**全員のラベルを使って選んでから** 交差検証をすると、何が起こるでしょうか。脳のデータの代わりに、**乱数** で試します。

**考えてみましょう**：脳と関係のない、ただの乱数で、どのくらい当たるでしょうか。

In [ ]:
noise = rng.standard_normal(X.shape).astype(np.float32)  # 脳と関係のない、でたらめな数の表（871 人 × 19,900）
chosen = SelectKBest(f_classif, k=500).fit(noise, y).get_support()  # 全員のラベルで、差の大きい 500 個を選ぶ
K_noise = noise[:, chosen] @ noise[:, chosen].T
predicted_noise = cross_val_predict(svm, K_noise, y, cv=ten_fold)
print("random numbers, chosen with everyone's labels: accuracy", (predicted_noise == y).mean().round(3))

**ただの乱数で 0.9 を超えました。** 選ぶときに、テストする人の答えも見てしまったためです（**リーク**、または二度使い）。
19,900 個もあれば、たまたまラベルとそろった乱数がいくらでも見つかります。01 の 8 章の「いちばん強い場所を選んで見ると、
合って見えるのは当然」と同じ問題です。

選ぶところも学習の一部なので、交差検証の **内側で**、学習に使う人のラベルだけで選びます。次のセルで、そのようにやり直します。

In [ ]:
choose_then_learn = make_pipeline(SelectKBest(f_classif, k=500), SVC(kernel="linear", C=0.001))  # 選ぶ特徴が毎回変わるので、内積の表は使わない
predicted_inside = cross_val_predict(choose_then_learn, noise, y, cv=ten_fold)
print("random numbers, chosen inside the cross-validation: accuracy", (predicted_inside == y).mean().round(3))

偶然の水準に戻りました。

> **くわしく**（時間があれば）
>
> - リーク（二度使い）の問題は、Kriegeskorte et al.（2009）がくわしく論じています。
> - scikit-learn の `make_pipeline` で「選ぶ → 学習する」を 1 つにまとめると、交差検証の中で毎回選び直してくれます。

5 章の C ＝ 0.001 も、試して選んだ値でした。この教材を作るときには、前処理のバージョンや分類器、C の値を、少なくとも 12 通り
試しました（最後の「補足」）。**何を、いくつ試したかを、すべて書いておく** ことが大切です。

ここまでで、5 章の正解率と p 値、6 章の 2 つの正解率が出ていれば、**ゴール 5** 達成です。

> **くわしく**（時間があれば）
>
> - 試して選んだ値は、交差検証の内側で選び直すこともできます（**入れ子の交差検証**）。この教材を作るときに、C だけを内側で
>   選び直しても、正解率はほとんど変わりませんでした。ただし、C 以外に試したこと（前処理のバージョンや分類器）まで含めて
>   検証し直したことにはなりません。
> - 5 章の並べ替え検定の p 値は、C を 0.001 に固定して計算したもので、試して選んだことは入っていません。試した 12 通りのうち、
>   ほとんど学習できなかった C ＝ 0.0001 を除くと、正解率はどれも 0.67〜0.70 でした。

## 7. 人数を集めると

### 7.1 学習に使う人数

人数を集めると、何がよくなるのでしょうか。無作為に選んだ 150 人をテスト用に残し、残りから選んだ 20〜700 人で学習して当てます。

In [ ]:
n_train_list = [20, 50, 100, 200, 400, 700]  # 学習に使う人数
mean_accuracy = []
for n_train in n_train_list:
    accuracies = []
    for repeat in range(10):
        order = rng.permutation(len(y))
        test, train = order[:150], order[150:150 + n_train]
        svm.fit(K[np.ix_(train, train)], y[train])
        accuracies.append((svm.predict(K[np.ix_(test, train)]) == y[test]).mean())
    mean_accuracy.append(np.mean(accuracies))
    print(f"{n_train:3d} people for training -> accuracy {np.mean(accuracies):.2f}")

plt.plot(n_train_list, mean_accuracy, marker="o")
plt.axhline((y == "TD").mean(), color="gray", linestyle="--", label="always answering the larger group (TD)")
plt.xlabel("people used for training")
plt.ylabel("accuracy on 150 other people")
plt.ylim(0.4, 0.8)
plt.legend(loc="lower right")
plt.show()

数十人で学習しても、多いほうを答え続けるのとほとんど変わりません。人数を増やすほど当たるようになり、数百人で 0.7 くらいに
なります。小さいけれども本物の違いを捉えるには、多くの人のデータが要ります。多くの施設がデータを持ち寄って公開する意味は、
ここにあります（02 では、19 人で学習して 0.96 でした）。

> **くわしく**（時間があれば）
>
> - 人の選び方を変えて、10 回ずつ繰り返しています。400 人と 700 人の違いは、この繰り返しの
>   数では、ばらつきの範囲です。
> - Abraham et al.（2017）も、人数を増やすほど当たるようになると報告しています。
> - 02 の 8 章の演習 1 では、5 人で学習しても 0.9 以上当たります。

### 7.2 ASD 20 人・定型 20 人の研究

ASD 20 人・定型 20 人を集めた研究は、よくあります。871 人から 20 人ずつ選んで「小さな研究」を 200 回作り、
それぞれの研究の中で交差検証した正解率（灰色）と、その 40 人で学習した分類器を残りの 831 人に当てたときの、実際の当たり方
（オレンジ）を比べます。

In [ ]:
reported, actual = [], []
for repeat in range(200):
    chosen_people = np.concatenate([rng.choice(np.where(y == group)[0], 20, replace=False) for group in ["ASD", "TD"]])
    others = np.setdiff1d(np.arange(len(y)), chosen_people)  # 選ばれなかった人
    K_small = K[np.ix_(chosen_people, chosen_people)]
    five_fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=repeat)
    reported.append(cross_val_score(svm, K_small, y[chosen_people], cv=five_fold).mean())  # 研究の中の交差検証
    svm.fit(K_small, y[chosen_people])
    actual.append(balanced_accuracy_score(y[others], svm.predict(K[np.ix_(others, chosen_people)])))  # ほかの人に当てる
reported, actual = np.array(reported), np.array(actual)
for name, values in [("reported (cross-validation in the study)", reported), ("actual (on the other 831 people)", actual)]:
    print(f"{name}: mean {values.mean():.2f} / 5-95%: {np.percentile(values, [5, 95]).round(2)}")
print("studies reporting 0.70 or higher:", f"{np.mean(reported >= 0.7):.0%}")

In [ ]:
bins = np.linspace(0.2, 0.9, 36)
plt.hist(reported, bins=bins, alpha=0.6, color="gray", label="reported (cross-validation in the study)")
plt.hist(actual, bins=bins, alpha=0.6, color="tab:orange", label="actual (on the other 831 people)")
plt.axvline(0.5, color="black", linestyle="--")
plt.xlabel("accuracy of one study (20 ASD + 20 TD)")
plt.ylabel("studies")
plt.legend(loc="upper left")
plt.show()

実際の当たり方（オレンジ）は、どの研究でも 0.5〜0.6 くらいです（40 人では学習が足りません）。それなのに、研究の中の正解率
（灰色）は、0.4 を下回るものから 0.7 を超えるものまでばらつきます。小さな研究で 0.7 や 0.8 が報告されても、その分類器が
よく当たるとは限りません。

> **くわしく**（時間があれば）
>
> - 少人数の交差検証の誤差の大きさは、Varoquaux（2018）がくわしく論じています。
> - この図は「ラベルと脳が無関係なとき（偶然）」の分布ではありません。本物のラベルで、小さな研究を何度も繰り返したときの
>   ばらつきです。
> - 831 人は ASD と定型の人数が違うので、実際の当たり方は balanced accuracy で測っています。

## 8. まとめ

脳のつながりには、ASD の人と定型の人で、偶然ではない違いがありました。ただし違いは小さく（約 0.69）、正しい手順で
確かめることと、多くの人数が要りました。

**持ち帰りたいこと**（脳波や発話など、ほかの計測で ASD の人と定型の人を比べるときにも当てはまります）

1. **特徴を選ぶ、設定を選ぶなど、データを見て決めることは、すべて交差検証の内側で**。試したことはすべて書き、p 値を
   読むときにも考えに入れる。
2. **数十人の研究の正解率は、人の選び方だけで大きくぶれる**（7.2 では 0.4 以下〜0.7 以上）。人数を増やすか、幅を一緒に示す。
3. **群と、記録の条件や脳以外のものを重ねない**。施設・装置・時期が群で偏ると、その違いを当てただけで「ASD を当てた」
   ように見える。頭や体の動き（脳波なら瞬きや筋肉の電位）、年齢、性など、群で違いやすいものは測っておき、それだけで
   どのくらい当たるかを並べて示す。
4. **当てられることは、診断に使えることとも、仕組みが分かったこととも違う**。

**研究を計画するときに、自分に聞くこと**

- 何を知りたいのか（診断の手がかりになるか、群のあいだで脳のどこが違うか、症状や行動との関係か）
- 誰に通用してほしいのか。それに合わせて、誰のデータをテスト用に残すか
- 脳以外の何で当たりそうか（年齢・性、動き。脳波なら瞬きや筋肉の電位、発話なら話す量や課題）
- 何人いて、正解率の幅はどのくらいか

> **くわしく**（時間があれば）
>
> **調べたことと結果**（→ のあとは、気をつけたこと）
>
> - **脳のつながりから ASD か定型かを当てる**（871 人）：約 0.69。偶然より高い。ただし、作るときに設定を試して選んだことは、
>   一部しか検定に入っていない（6 章、補足）→ 試したことをすべて書く
> - **全員のラベルで特徴を選んでから検証**：乱数でも 0.9 以上 → 選ぶ・決めることは、すべて交差検証の内側で
> - **学習に使う人数**：数十人では、多いほうを答え続けるのとほとんど変わらず、数百人で 0.7 くらい → 人数を集める
> - **ASD 20 人・定型 20 人の研究を 200 回**：研究の中の正解率は 0.4 以下〜0.7 以上にばらつくが、実際の当たり方は 0.5〜0.6 くらい
>   → 少人数の交差検証の正解率は誤差が大きい
> - **施設と頭の動き**（5 章の「くわしく」）：同じ結合から施設が 17 択で約 0.75 当たるが、施設を丸ごと除いて検証しても約 0.69。
>   頭の動き・年齢・性だけでは 0.565 → 群と施設を重ねない。脳以外の情報だけでどのくらい当たるかを並べる

## 9. 自分で試してみよう

セルを書き換えて試せます。時間があれば試してください（持ち帰って試してもかまいません）。

> **くわしく**（時間があれば）
>
> 1. 5 章の `C=0.001` を `1.0` や `0.0001` にして、そのセルから下を順に実行し直す。0.0001 では、何を答えるようになるでしょうか。
> 2. 7.2 の 20（`20, replace=False`）を 50 や 100 にして、そのセルと次のセルを実行し直す。灰色とオレンジのばらつきは
>    どう変わるでしょうか。
> 3. 2 章の `strategy = "filt_noglobal"` を `"filt_global"`（脳全体の平均の信号も取り除いたバージョン）にして、メニューの
>    「ランタイム」→「すべてのセルを実行」でやり直す（もう一度、約 340 MB を取ってきます）。5 章と 6 章の結果は、
>    どう変わるでしょうか。

## 10. この先へ

ASD の研究で、この先に進むときの方法とデータです。

> **くわしく**（時間があれば）
>
> - **施設の違いを取り除く**：施設ごとの平均やばらつきの違いを統計的に取り除く方法（ComBat など。Yu et al., 2018）。
>   ただし、群と施設が重なっていると、群の違いまで取り除いてしまうおそれがあります。
> - **別のデータで確かめる**：日本の施設のデータで作った分類器を、米国と日本の別の人たちで確かめた研究があります
>   （Yahata et al., 2016）。
> - **ABIDE II**：ABIDE I のあとに集められた、さらに 1,000 人ほどのデータ。
> - **診断の 2 択ではなく、症状の強さ（ADOS などの得点）を予測する**、**同じ年齢・性の定型の人からどのくらい外れているかを見る**
>   など、ASD の人どうしの違いを扱う方法。

## 11. 使った環境を記録する

今回の解析に使った Python と道具のバージョンを表示します。

In [ ]:
import sys

import nilearn
import scipy
import sklearn

print("Python:", sys.version.split()[0])
print("nilearn:", nilearn.__version__)
print("scikit-learn:", sklearn.__version__)
print("scipy:", scipy.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

## 補足：この教材を作るときに試したこと

6 章で触れた、試したことの一覧と、それを考えに入れた確かめ方です。

> **くわしく**（時間があれば）
>
> この教材を作るときには、全員のデータで、少なくとも次の 12 通りを試しました（このときの交差検証は、5 章の「くわしく」の、
> 施設を丸ごと除く形です）。
>
> - 帯域フィルタありのバージョン（5 章と同じ）：SVM（C ＝ 1、0.01、0.001、0.0001）、リッジ回帰（強さ 3 通り）、ロジスティック回帰
> - 帯域フィルタなしのバージョン：SVM（C ＝ 1、0.001）、ロジスティック回帰
> - 全脳の平均の信号を取り除くバージョン：SVM（C ＝ 0.001）
>
> ほかに、C を内側で選ぶ形や、1 つの施設のデータだけで試したものもあります。上の 12 通りのうち、ほとんど学習できなかった
> C ＝ 0.0001 を除くと、正解率はどれも 0.67〜0.70 でした。
>
> 試した数で p 値を大きくする、いちばん単純な補正（**Bonferroni の補正**。p 値に試した数を掛ける）をすると、12 通りとして
> 数えても 0.005 × 12 ＝ 0.06 で、0.05 を超えてしまいます。これは、200 回の並べ替えでは p 値を 0.005 より小さく示せないため
> です。つまり、200 回の並べ替えだけでは、これだけ試したあとでも偶然より高いと言えるかを確かめられません。
>
> そこで、ノートブックの外で、帯域フィルタの有無の 2 通り ×（SVM の C 4 通り＋リッジ回帰 3 通り）の 14 通りから、いちばん
> 良いものを選ぶところまで含めた並べ替え検定もしました（100 回）。混ぜたときの最大は 0.57 で、本物の 0.69 には一度も
> 届きませんでした（p ≈ 0.01。100 回で出せるいちばん小さい値）。ただし、ロジスティック回帰と、全脳の平均の信号を取り除くバージョンは、
> この検定に入っていません。試したことをすべて含めた補正は、確かめきれていません。この検定のスクリプトは、この教材の
> リポジトリの `tools/abide_selection_null.py` にあります。

## データと道具、方法の出典

- データ：Di Martino, A., et al. (2014). The autism brain imaging data exchange: towards a large-scale evaluation of the
  intrinsic brain architecture in autism. *Molecular Psychiatry*, 19, 659–667. https://doi.org/10.1038/mp.2013.78
  （http://fcon_1000.projects.nitrc.org/indi/abide/abide_I.html 。利用条件は、このページの Usage Agreement：
  Creative Commons 表示-非営利-継承 3.0 https://creativecommons.org/licenses/by-nc-sa/3.0/ ）
- 前処理済みのデータ：Craddock, C., Benhajali, Y., Chu, C., Chouinard, F., Evans, A., Jakab, A., Khundrakpam, B. S.,
  Lewis, J. D., Li, Q., Milham, M., Yan, C., & Bellec, P. (2013). The Neuro Bureau Preprocessing Initiative: open sharing of
  preprocessed neuroimaging data and derivatives. *Frontiers in Neuroinformatics*, Conference Abstract: Neuroinformatics 2013.
  https://doi.org/10.3389/conf.fninf.2013.09.00041 （http://preprocessed-connectomes-project.org/abide/ 。
  このデータを使うときは、この抄録を引くよう求められています）
- 領域の地図（CC200）：Craddock, R. C., James, G. A., Holtzheimer, P. E., Hu, X. P., & Mayberg, H. S. (2012). A whole brain
  fMRI atlas generated via spatially constrained spectral clustering. *Human Brain Mapping*, 33, 1914–1928.
  https://doi.org/10.1002/hbm.21333
- ABIDE での分類：Nielsen, J. A., et al. (2013). Multisite functional connectivity MRI classification of autism: ABIDE results.
  *Frontiers in Human Neuroscience*, 7, 599. https://doi.org/10.3389/fnhum.2013.00599
- ABIDE での分類：Abraham, A., et al. (2017). Deriving reproducible biomarkers from multi-site resting-state data:
  An Autism-based example. *NeuroImage*, 147, 736–745. https://doi.org/10.1016/j.neuroimage.2016.10.045
- ABIDE での分類：Heinsfeld, A. S., Franco, A. R., Craddock, R. C., Buchweitz, A., & Meneguzzi, F. (2018). Identification of
  autism spectrum disorder using deep learning and the ABIDE dataset. *NeuroImage: Clinical*, 17, 16–23.
  https://doi.org/10.1016/j.nicl.2017.08.017
- 頭の動き：Power, J. D., Barnes, K. A., Snyder, A. Z., Schlaggar, B. L., & Petersen, S. E. (2012). Spurious but systematic
  correlations in functional connectivity MRI networks arise from subject motion. *NeuroImage*, 59, 2142–2154.
  https://doi.org/10.1016/j.neuroimage.2011.10.018
- リーク：Kriegeskorte, N., Simmons, W. K., Bellgowan, P. S. F., & Baker, C. I. (2009). Circular analysis in systems
  neuroscience: the dangers of double dipping. *Nature Neuroscience*, 12, 535–540. https://doi.org/10.1038/nn.2303
- 少人数の正解率の幅：Varoquaux, G. (2018). Cross-validation failure: Small sample sizes lead to large error bars.
  *NeuroImage*, 180, 68–77. https://doi.org/10.1016/j.neuroimage.2017.06.061
- 並べ替え検定：Ojala, M., & Garriga, G. C. (2010). Permutation tests for studying classifier performance.
  *Journal of Machine Learning Research*, 11, 1833–1863.
- 施設の違いを取り除く：Yu, M., et al. (2018). Statistical harmonization corrects site effects in functional connectivity
  measurements from multi-site fMRI data. *Human Brain Mapping*, 39, 4213–4227. https://doi.org/10.1002/hbm.24241
- 別のデータで確かめる：Yahata, N., et al. (2016). A small number of abnormal brain connections predicts adult autism spectrum
  disorder. *Nature Communications*, 7, 11254. https://doi.org/10.1038/ncomms11254
- 解析：Nilearn https://nilearn.github.io/ 、scikit-learn https://scikit-learn.org/